In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)



In [2]:
import torchvision.models as models
import torch.utils.model_zoo as model_zoo
import torch.nn.functional as F
import types
import re
import torch.nn as nn  # ensure nn is available for nn.Identity used in modify_densenets

__all__ = [
    "alexnet",
    "densenet121",
    "densenet169",
    "densenet201",
    "densenet161",
    "resnet18",
    "resnet34",
    "resnet50",
    "resnet101",
    "resnet152",
    "inceptionv3",
    "squeezenet1_0",
    "squeezenet1_1",
    "vgg11",
    "vgg11_bn",
    "vgg13",
    "vgg13_bn",
    "vgg16",
    "vgg16_bn",
    "vgg19_bn",
    "vgg19",
]

model_urls = {
    "alexnet": "https://download.pytorch.org/models/alexnet-owt-4df8aa71.pth",
    "densenet121": "http://data.lip6.fr/cadene/pretrainedmodels/densenet121-fbdb23505.pth",
    "densenet169": "http://data.lip6.fr/cadene/pretrainedmodels/densenet169-f470b90a4.pth",
    "densenet201": "http://data.lip6.fr/cadene/pretrainedmodels/densenet201-5750cbb1e.pth",
    "densenet161": "http://data.lip6.fr/cadene/pretrainedmodels/densenet161-347e6b360.pth",
    "inceptionv3": "https://download.pytorch.org/models/inception_v3_google-1a9a5a14.pth",
    "resnet18": "https://download.pytorch.org/models/resnet18-5c106cde.pth",
    "resnet34": "https://download.pytorch.org/models/resnet34-333f7ec4.pth",
    "resnet50": "https://download.pytorch.org/models/resnet50-19c8e357.pth",
    "resnet101": "https://download.pytorch.org/models/resnet101-5d3b4d8f.pth",
    "resnet152": "https://download.pytorch.org/models/resnet152-b121ed2d.pth",
    "squeezenet1_0": "https://download.pytorch.org/models/squeezenet1_0-a815701f.pth",
    "squeezenet1_1": "https://download.pytorch.org/models/squeezenet1_1-f364aa15.pth",
    "vgg11": "https://download.pytorch.org/models/vgg11-bbd30ac9.pth",
    "vgg13": "https://download.pytorch.org/models/vgg13-c768596a.pth",
    "vgg16": "https://download.pytorch.org/models/vgg16-397923af.pth",
    "vgg19": "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth",
    "vgg11_bn": "https://download.pytorch.org/models/vgg11_bn-6002323d.pth",
    "vgg13_bn": "https://download.pytorch.org/models/vgg13_bn-abd245e5.pth",
    "vgg16_bn": "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth",
    "vgg19_bn": "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth",
}

input_sizes = {}
means = {}
stds = {}

for model_name in __all__:
    input_sizes[model_name] = [3, 224, 224]
    means[model_name] = [0.485, 0.456, 0.406]
    stds[model_name] = [0.229, 0.224, 0.225]

for model_name in ["inceptionv3"]:
    input_sizes[model_name] = [3, 299, 299]
    means[model_name] = [0.5, 0.5, 0.5]
    stds[model_name] = [0.5, 0.5, 0.5]

tv_pretrained_settings = {}

for model_name in __all__:
    tv_pretrained_settings[model_name] = {
        "imagenet": {
            "url": model_urls[model_name],
            "input_space": "RGB",
            "input_size": input_sizes[model_name],
            "input_range": [0, 1],
            "mean": means[model_name],
            "std": stds[model_name],
            "num_classes": 1000,
        }
    }


def update_state_dict(state_dict):
    pattern = re.compile(
        r"^(.*denselayer\d+\.(?:norm|relu|conv))\.((?:[12])\.(?:weight|bias|running_mean|running_var))$"
    )
    for key in list(state_dict.keys()):
        res = pattern.match(key)
        if res:
            new_key = res.group(1) + res.group(2)
            state_dict[new_key] = state_dict[key]
            del state_dict[key]
    return state_dict


def load_pretrained(model, num_classes, settings):
    """
    Fully offline-safe pretrained loading:
    use torchvision built-in weight enums when available; otherwise proceed without weights.
    """
    assert (
        num_classes == settings["num_classes"]
    ), "num_classes should be {}, but is {}".format(
        settings["num_classes"], num_classes
    )

    state_dict = None
    try:
        if isinstance(model, models.DenseNet) and model.features.conv0.in_channels == 3:
            from torchvision.models import (
                DenseNet121_Weights,
                DenseNet169_Weights,
                DenseNet201_Weights,
                DenseNet161_Weights,
            )

            in_feats = model.classifier.in_features
            if in_feats == 1024:
                state_dict = models.densenet121(
                    weights=DenseNet121_Weights.DEFAULT
                ).state_dict()
            elif in_feats == 1664:
                state_dict = models.densenet169(
                    weights=DenseNet169_Weights.DEFAULT
                ).state_dict()
            elif in_feats == 1920:
                state_dict = models.densenet201(
                    weights=DenseNet201_Weights.DEFAULT
                ).state_dict()
            else:
                state_dict = models.densenet161(
                    weights=DenseNet161_Weights.DEFAULT
                ).state_dict()

        elif isinstance(model, models.ResNet):
            from torchvision.models import (
                ResNet18_Weights,
                ResNet34_Weights,
                ResNet50_Weights,
                ResNet101_Weights,
                ResNet152_Weights,
            )

            layers = [
                len(model.layer1),
                len(model.layer2),
                len(model.layer3),
                len(model.layer4),
            ]
            if layers == [2, 2, 2, 2]:
                state_dict = models.resnet18(
                    weights=ResNet18_Weights.DEFAULT
                ).state_dict()
            elif layers == [3, 4, 6, 3]:
                state_dict = models.resnet50(
                    weights=ResNet50_Weights.DEFAULT
                ).state_dict()
            elif layers == [3, 4, 23, 3]:
                state_dict = models.resnet101(
                    weights=ResNet101_Weights.DEFAULT
                ).state_dict()
            elif layers == [3, 8, 36, 3]:
                state_dict = models.resnet152(
                    weights=ResNet152_Weights.DEFAULT
                ).state_dict()
            else:
                state_dict = models.resnet34(
                    weights=ResNet34_Weights.DEFAULT
                ).state_dict()
    except Exception:
        state_dict = None

    if state_dict is None:
        state_dict = {}

    if len(state_dict) > 0:
        state_dict = update_state_dict(state_dict)
        model.load_state_dict(state_dict, strict=False)

    model.input_space = settings["input_space"]
    model.input_size = settings["input_size"]
    model.input_range = settings["input_range"]
    model.mean = settings["mean"]
    model.std = settings["std"]
    return model


def modify_alexnet(model):
    model._features = model.features
    del model.features
    model.dropout0 = model.classifier[0]
    model.linear0 = model.classifier[1]
    model.relu0 = model.classifier[2]
    model.dropout1 = model.classifier[3]
    model.linear1 = model.classifier[4]
    model.relu1 = model.classifier[5]
    model.last_linear = model.classifier[6]
    del model.classifier

    def features(self, input):
        x = self._features(input)
        x = x.view(x.size(0), 256 * 6 * 6)
        x = self.dropout0(x)
        x = self.linear0(x)
        x = self.relu0(x)
        x = self.dropout1(x)
        x = self.linear1(x)
        return x

    def logits(self, features):
        x = self.relu1(features)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

    model.features = types.MethodType(features, model)
    model.logits = types.MethodType(logits, model)
    model.forward = types.MethodType(forward, model)
    return model


def alexnet(num_classes=1000, pretrained="imagenet"):
    model = models.alexnet(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["alexnet"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_alexnet(model)
    return model


def modify_densenets(model):
    model.last_linear = model.classifier
    model.classifier = nn.Identity()

    def logits(self, features):
        x = F.relu(features, inplace=True)
        if hasattr(self, "avg_pool") and self.avg_pool is not None:
            x = self.avg_pool(x)
        else:
            x = F.adaptive_avg_pool2d(x, (1, 1))
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

    model.logits = types.MethodType(logits, model)
    model.forward = types.MethodType(forward, model)
    return model


def densenet121(num_classes=1000, pretrained="imagenet"):
    model = models.densenet121(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["densenet121"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_densenets(model)
    return model


def densenet169(num_classes=1000, pretrained="imagenet"):
    model = models.densenet169(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["densenet169"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_densenets(model)
    return model


def densenet201(num_classes=1000, pretrained="imagenet"):
    model = models.densenet201(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["densenet201"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_densenets(model)
    return model


def densenet161(num_classes=1000, pretrained="imagenet"):
    model = models.densenet161(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["densenet161"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_densenets(model)
    return model


def inceptionv3(num_classes=1000, pretrained="imagenet"):
    model = models.inception_v3(weights=None, aux_logits=True)
    if pretrained is not None:
        settings = tv_pretrained_settings["inceptionv3"][pretrained]
        model = load_pretrained(model, num_classes, settings)

    model.last_linear = model.fc
    del model.fc

    def features(self, input):
        x = self.Conv2d_1a_3x3(input)
        x = self.Conv2d_2a_3x3(x)
        x = self.Conv2d_2b_3x3(x)
        x = F.max_pool2d(x, kernel_size=3, stride=2)
        x = self.Conv2d_3b_1x1(x)
        x = self.Conv2d_4a_3x3(x)
        x = F.max_pool2d(x, kernel_size=3, stride=2)
        x = self.Mixed_5b(x)
        x = self.Mixed_5c(x)
        x = self.Mixed_5d(x)
        x = self.Mixed_6a(x)
        x = self.Mixed_6b(x)
        x = self.Mixed_6c(x)
        x = self.Mixed_6d(x)
        x = self.Mixed_6e(x)
        if self.training and self.aux_logits:
            self._out_aux = self.AuxLogits(x)
        x = self.Mixed_7a(x)
        x = self.Mixed_7b(x)
        x = self.Mixed_7c(x)
        return x

    def logits(self, features):
        x = F.avg_pool2d(features, kernel_size=8)
        x = F.dropout(x, training=self.training)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        if self.training and self.aux_logits:
            aux = self._out_aux
            self._out_aux = None
            return x, aux
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

    model.features = types.MethodType(features, model)
    model.logits = types.MethodType(logits, model)
    model.forward = types.MethodType(forward, model)
    return model


def modify_resnets(model):
    model.last_linear = model.fc
    model.fc = None

    def features(self, input):
        x = self.conv1(input)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def logits(self, features):
        x = self.avgpool(features)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

    model.features = types.MethodType(features, model)
    model.logits = types.MethodType(logits, model)
    model.forward = types.MethodType(forward, model)
    return model


def resnet18(num_classes=1000, pretrained="imagenet"):
    model = models.resnet18(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["resnet18"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_resnets(model)
    return model


def resnet34(num_classes=1000, pretrained="imagenet"):
    model = models.resnet34(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["resnet34"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_resnets(model)
    return model


def resnet50(num_classes=1000, pretrained="imagenet"):
    model = models.resnet50(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["resnet50"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_resnets(model)
    return model


def resnet101(num_classes=1000, pretrained="imagenet"):
    model = models.resnet101(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["resnet101"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_resnets(model)
    return model


def resnet152(num_classes=1000, pretrained="imagenet"):
    model = models.resnet152(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["resnet152"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_resnets(model)
    return model


def modify_squeezenets(model):
    model.dropout = model.classifier[0]
    model.last_conv = model.classifier[1]
    model.relu = model.classifier[2]
    model.avgpool = model.classifier[3]
    del model.classifier

    def logits(self, features):
        x = self.dropout(features)
        x = self.last_conv(x)
        x = self.relu(x)
        x = self.avgpool(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

    model.logits = types.MethodType(logits, model)
    model.forward = types.MethodType(forward, model)
    return model


def squeezenet1_0(num_classes=1000, pretrained="imagenet"):
    model = models.squeezenet1_0(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["squeezenet1_0"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_squeezenets(model)
    return model


def squeezenet1_1(num_classes=1000, pretrained="imagenet"):
    model = models.squeezenet1_1(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["squeezenet1_1"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_squeezenets(model)
    return model


def modify_vggs(model):
    model._features = model.features
    del model.features
    model.linear0 = model.classifier[0]
    model.relu0 = model.classifier[1]
    model.dropout0 = model.classifier[2]
    model.linear1 = model.classifier[3]
    model.relu1 = model.classifier[4]
    model.dropout1 = model.classifier[5]
    model.last_linear = model.classifier[6]
    del model.classifier

    def features(self, input):
        x = self._features(input)
        x = x.view(x.size(0), -1)
        x = self.linear0(x)
        x = self.relu0(x)
        x = self.dropout0(x)
        x = self.linear1(x)
        return x

    def logits(self, features):
        x = self.relu1(features)
        x = self.dropout1(x)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

    model.features = types.MethodType(features, model)
    model.logits = types.MethodType(logits, model)
    model.forward = types.MethodType(forward, model)
    return model


def vgg11(num_classes=1000, pretrained="imagenet"):
    model = models.vgg11(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["vgg11"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_vggs(model)
    return model


def vgg11_bn(num_classes=1000, pretrained="imagenet"):
    model = models.vgg11_bn(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["vgg11_bn"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_vggs(model)
    return model


def vgg13(num_classes=1000, pretrained="imagenet"):
    model = models.vgg13(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["vgg13"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_vggs(model)
    return model


def vgg13_bn(num_classes=1000, pretrained="imagenet"):
    model = models.vgg13_bn(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["vgg13_bn"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_vggs(model)
    return model


def vgg16(num_classes=1000, pretrained="imagenet"):
    model = models.vgg16(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["vgg16"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_vggs(model)
    return model


def vgg16_bn(num_classes=1000, pretrained="imagenet"):
    model = models.vgg16_bn(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["vgg16_bn"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_vggs(model)
    return model


def vgg19(num_classes=1000, pretrained="imagenet"):
    model = models.vgg19(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["vgg19"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_vggs(model)
    return model


def vgg19_bn(num_classes=1000, pretrained="imagenet"):
    model = models.vgg19_bn(weights=None)
    if pretrained is not None:
        settings = tv_pretrained_settings["vgg19_bn"][pretrained]
        model = load_pretrained(model, num_classes, settings)
    model = modify_vggs(model)
    return model




In [3]:
"""
ResNet code gently borrowed from
https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py
"""
from collections import OrderedDict
import math

import torch
import torch.nn as nn
from torch.utils import model_zoo

__all__ = [
    "SENet",
    "senet154",
    "se_resnet50",
    "se_resnet101",
    "se_resnet152",
    "se_resnext50_32x4d",
    "se_resnext101_32x4d",
]

pretrained_settings = {
    "senet154": {
        "imagenet": {
            "url": "http://data.lip6.fr/cadene/pretrainedmodels/senet154-c7b49a05.pth",
            "input_space": "RGB",
            "input_size": [3, 224, 224],
            "input_range": [0, 1],
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
            "num_classes": 1000,
        }
    },
    "se_resnet50": {
        "imagenet": {
            "url": "http://data.lip6.fr/cadene/pretrainedmodels/se_resnet50-ce0d4300.pth",
            "input_space": "RGB",
            "input_size": [3, 224, 224],
            "input_range": [0, 1],
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
            "num_classes": 1000,
        }
    },
    "se_resnet101": {
        "imagenet": {
            "url": "http://data.lip6.fr/cadene/pretrainedmodels/se_resnet101-7e38fcc6.pth",
            "input_space": "RGB",
            "input_size": [3, 224, 224],
            "input_range": [0, 1],
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
            "num_classes": 1000,
        }
    },
    "se_resnet152": {
        "imagenet": {
            "url": "http://data.lip6.fr/cadene/pretrainedmodels/se_resnet152-d17c99b7.pth",
            "input_space": "RGB",
            "input_size": [3, 224, 224],
            "input_range": [0, 1],
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
            "num_classes": 1000,
        }
    },
    "se_resnext50_32x4d": {
        "imagenet": {
            "url": "http://data.lip6.fr/cadene/pretrainedmodels/se_resnext50_32x4d-a260b3a4.pth",
            "input_space": "RGB",
            "input_size": [3, 224, 224],
            "input_range": [0, 1],
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
            "num_classes": 1000,
        }
    },
    "se_resnext101_32x4d": {
        "imagenet": {
            "url": "http://data.lip6.fr/cadene/pretrainedmodels/se_resnext101_32x4d-3b2fe3d8.pth",
            "input_space": "RGB",
            "input_size": [3, 224, 224],
            "input_range": [0, 1],
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
            "num_classes": 1000,
        }
    },
}


class SEModule(nn.Module):
    def __init__(self, channels, reduction):
        super(SEModule, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1, padding=0)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1, padding=0)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.avg_pool(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return module_input * x


class Bottleneck(nn.Module):
    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out = self.se_module(out) + residual
        out = self.relu(out)

        return out


class SEBottleneck(Bottleneck):
    expansion = 4

    def __init__(self, inplanes, planes, groups, reduction, stride=1, downsample=None):
        super(SEBottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes * 2, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes * 2)
        self.conv2 = nn.Conv2d(
            planes * 2,
            planes * 4,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=groups,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(planes * 4)
        self.conv3 = nn.Conv2d(planes * 4, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SEResNetBottleneck(Bottleneck):
    expansion = 4

    def __init__(self, inplanes, planes, groups, reduction, stride=1, downsample=None):
        super(SEResNetBottleneck, self).__init__()
        self.conv1 = nn.Conv2d(
            inplanes, planes, kernel_size=1, bias=False, stride=stride
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, padding=1, groups=groups, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SEResNeXtBottleneck(Bottleneck):
    expansion = 4

    def __init__(
        self,
        inplanes,
        planes,
        groups,
        reduction,
        stride=1,
        downsample=None,
        base_width=4,
    ):
        super(SEResNeXtBottleneck, self).__init__()
        width = math.floor(planes * (base_width / 64)) * groups
        self.conv1 = nn.Conv2d(inplanes, width, kernel_size=1, bias=False, stride=1)
        self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(
            width,
            width,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=groups,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SENet(nn.Module):
    def __init__(
        self,
        block,
        layers,
        groups,
        reduction,
        dropout_p=0.2,
        inplanes=128,
        input_3x3=True,
        downsample_kernel_size=3,
        downsample_padding=1,
        num_classes=1000,
    ):
        super(SENet, self).__init__()
        self.inplanes = inplanes
        if input_3x3:
            layer0_modules = [
                ("conv1", nn.Conv2d(3, 64, 3, stride=2, padding=1, bias=False)),
                ("bn1", nn.BatchNorm2d(64)),
                ("relu1", nn.ReLU(inplace=True)),
                ("conv2", nn.Conv2d(64, 64, 3, stride=1, padding=1, bias=False)),
                ("bn2", nn.BatchNorm2d(64)),
                ("relu2", nn.ReLU(inplace=True)),
                ("conv3", nn.Conv2d(64, inplanes, 3, stride=1, padding=1, bias=False)),
                ("bn3", nn.BatchNorm2d(inplanes)),
                ("relu3", nn.ReLU(inplace=True)),
            ]
        else:
            layer0_modules = [
                (
                    "conv1",
                    nn.Conv2d(
                        3, inplanes, kernel_size=7, stride=2, padding=3, bias=False
                    ),
                ),
                ("bn1", nn.BatchNorm2d(inplanes)),
                ("relu1", nn.ReLU(inplace=True)),
            ]
        layer0_modules.append(("pool", nn.MaxPool2d(3, stride=2, ceil_mode=True)))
        self.layer0 = nn.Sequential(OrderedDict(layer0_modules))
        self.layer1 = self._make_layer(
            block,
            planes=64,
            blocks=layers[0],
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=1,
            downsample_padding=0,
        )
        self.layer2 = self._make_layer(
            block,
            planes=128,
            blocks=layers[1],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding,
        )
        self.layer3 = self._make_layer(
            block,
            planes=256,
            blocks=layers[2],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding,
        )
        self.layer4 = self._make_layer(
            block,
            planes=512,
            blocks=layers[3],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding,
        )
        self.avg_pool = nn.AvgPool2d(7, stride=1)
        self.dropout = nn.Dropout(dropout_p) if dropout_p is not None else None
        self.last_linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(
        self,
        block,
        planes,
        blocks,
        groups,
        reduction,
        stride=1,
        downsample_kernel_size=1,
        downsample_padding=0,
    ):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.inplanes,
                    planes * block.expansion,
                    kernel_size=downsample_kernel_size,
                    stride=stride,
                    padding=downsample_padding,
                    bias=False,
                ),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = [block(self.inplanes, planes, groups, reduction, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups, reduction))

        return nn.Sequential(*layers)

    def features(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def logits(self, x):
        x = self.avg_pool(x)
        if self.dropout is not None:
            x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, x):
        x = self.features(x)
        x = self.logits(x)
        return x


def initialize_pretrained_model(model, num_classes, settings):
    """
    Offline-safe init: try loading URL; if offline/unavailable, continue (keeps model runnable).
    """
    assert (
        num_classes == settings["num_classes"]
    ), "num_classes should be {}, but is {}".format(
        settings["num_classes"], num_classes
    )
    try:
        state = model_zoo.load_url(settings["url"])
        model.load_state_dict(state, strict=False)
    except Exception:
        pass

    model.input_space = settings["input_space"]
    model.input_size = settings["input_size"]
    model.input_range = settings["input_range"]
    model.mean = settings["mean"]
    model.std = settings["std"]


def senet154(num_classes=1000, pretrained="imagenet"):
    model = SENet(
        SEBottleneck,
        [3, 8, 36, 3],
        groups=64,
        reduction=16,
        dropout_p=0.2,
        num_classes=num_classes,
    )
    if pretrained is not None:
        settings = pretrained_settings["senet154"][pretrained]
        initialize_pretrained_model(model, num_classes, settings)
    return model


def se_resnet50(num_classes=1000, pretrained="imagenet"):
    model = SENet(
        SEResNetBottleneck,
        [3, 4, 6, 3],
        groups=1,
        reduction=16,
        dropout_p=None,
        inplanes=64,
        input_3x3=False,
        downsample_kernel_size=1,
        downsample_padding=0,
        num_classes=num_classes,
    )
    if pretrained is not None:
        settings = pretrained_settings["se_resnet50"][pretrained]
        initialize_pretrained_model(model, num_classes, settings)
    return model


def se_resnet101(num_classes=1000, pretrained="imagenet"):
    model = SENet(
        SEResNetBottleneck,
        [3, 4, 23, 3],
        groups=1,
        reduction=16,
        dropout_p=None,
        inplanes=64,
        input_3x3=False,
        downsample_kernel_size=1,
        downsample_padding=0,
        num_classes=num_classes,
    )
    if pretrained is not None:
        settings = pretrained_settings["se_resnet101"][pretrained]
        initialize_pretrained_model(model, num_classes, settings)
    return model


def se_resnet152(num_classes=1000, pretrained="imagenet"):
    model = SENet(
        SEResNetBottleneck,
        [3, 8, 36, 3],
        groups=1,
        reduction=16,
        dropout_p=None,
        inplanes=64,
        input_3x3=False,
        downsample_kernel_size=1,
        downsample_padding=0,
        num_classes=num_classes,
    )
    if pretrained is not None:
        settings = pretrained_settings["se_resnet152"][pretrained]
        initialize_pretrained_model(model, num_classes, settings)
    return model


def se_resnext50_32x4d(num_classes=1000, pretrained="imagenet"):
    model = SENet(
        SEResNeXtBottleneck,
        [3, 4, 6, 3],
        groups=32,
        reduction=16,
        dropout_p=None,
        inplanes=64,
        input_3x3=False,
        downsample_kernel_size=1,
        downsample_padding=0,
        num_classes=num_classes,
    )
    if pretrained is not None:
        settings = pretrained_settings["se_resnext50_32x4d"][pretrained]
        initialize_pretrained_model(model, num_classes, settings)
    return model


def se_resnext101_32x4d(num_classes=1000, pretrained="imagenet"):
    model = SENet(
        SEResNeXtBottleneck,
        [3, 4, 23, 3],
        groups=32,
        reduction=16,
        dropout_p=None,
        inplanes=64,
        input_3x3=False,
        downsample_kernel_size=1,
        downsample_padding=0,
        num_classes=num_classes,
    )
    if pretrained is not None:
        settings = pretrained_settings["se_resnext101_32x4d"][pretrained]
        initialize_pretrained_model(model, num_classes, settings)
    return model




In [4]:
import os
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return gem(x, p=self.p, eps=self.eps)

    def __repr__(self):
        return (
            self.__class__.__name__
            + "("
            + "p="
            + "{:.4f}".format(self.p.data.tolist()[0])
            + ", "
            + "eps="
            + str(self.eps)
            + ")"
        )


def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


def get_se_resnet50_gem(pretrain):
    if pretrain == "imagenet":
        model = se_resnet50(num_classes=1000, pretrained="imagenet")
    else:
        model = se_resnet50(num_classes=1000, pretrained=None)
    model.avg_pool = GeM()
    model.last_linear = nn.Linear(2048, 1)
    return model


def get_densenet121_gem(pretrain):
    if pretrain == "imagenet":
        model = densenet121(num_classes=1000, pretrained="imagenet")
    else:
        model = densenet121(num_classes=1000, pretrained=None)
    model.avg_pool = GeM()
    model.last_linear = nn.Linear(1024, 1)
    return model




In [5]:
import os
from glob import glob

import pandas as pd
from PIL import Image, ImageFile
from torchvision import transforms
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ImageFile.LOAD_TRUNCATED_IMAGES = True

TEST_IMAGE_PATH = "/kaggle/input/aptos2019-blindness-detection/test_images"

test_df = pd.read_csv("/kaggle/input/aptos2019-blindness-detection/test.csv")
test_images = [
    os.path.join(TEST_IMAGE_PATH, f"{i}.png") for i in test_df["id_code"].tolist()
]




In [6]:
def make_predictions(model, test_images, transforms, size=256, device=device):
    predictions = []
    model.eval()
    with torch.inference_mode():
        for im_path in test_images:
            image = Image.open(im_path).convert("RGB")
            image = image.resize((size, size), resample=Image.BILINEAR)

            image = transforms(image).to(device)
            output = model(image.unsqueeze(0))
            output_flip = model(torch.flip(image.unsqueeze(0), dims=(3,)))
            final_prediction = (output.item() + output_flip.item()) / 2.0

            if not np.isfinite(final_prediction):
                final_prediction = 0.0

            predictions.append(
                (os.path.splitext(os.path.basename(im_path))[0], final_prediction)
            )
    return predictions


def _find_candidate_checkpoints():
    roots = ["/kaggle/input", "/kaggle/data/input"]
    pats = []
    for r in roots:
        if os.path.exists(r):
            pats.extend([os.path.join(r, "**", "*.pth"), os.path.join(r, "**", "*.pt")])
    files = []
    for p in pats:
        files.extend(glob(p, recursive=True))

    def rank(path):
        name = os.path.basename(path).lower()
        score = 0
        if "model_densenet121_bs64_30" in name:
            score -= 100
        if "model30_512" in name:
            score -= 50
        if "model10" in name:
            score -= 40
        if "densenet121" in name:
            score -= 20
        if "seresnet" in name or "se_resnet" in name:
            score -= 10
        try:
            sz = os.path.getsize(path)
        except Exception:
            sz = 0
        if sz < 1024 * 1024:
            score += 30
        return score

    files = sorted(set(files), key=rank)
    return files


def _stratified_sample_df(df, label_col, n, seed=2020):
    if n >= len(df):
        return df
    rng = np.random.RandomState(seed)
    vc = df[label_col].value_counts().sort_index()
    labels = vc.index.to_list()
    counts = vc.values.astype(np.float64)
    props = counts / counts.sum()
    base = np.floor(props * n).astype(int)

    base = np.maximum(base, 1)
    while base.sum() > n:
        k = int(rng.choice(np.where(base > 1)[0]))
        base[k] -= 1
    while base.sum() < n:
        k = int(rng.choice(len(base)))
        base[k] += 1

    parts = []
    for lab, k in zip(labels, base):
        sub = df[df[label_col] == lab]
        take = min(k, len(sub))
        parts.append(sub.sample(n=take, random_state=seed, replace=False))
    out = pd.concat(parts, axis=0)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out


def _fit_affine_calibration(
    model, train_df, train_img_dir, norm, size=256, device=device, max_images=512
):
    model.eval()
    xs, ys = [], []

    df_use = _stratified_sample_df(
        train_df, "diagnosis", n=min(max_images, len(train_df)), seed=2020
    )

    with torch.inference_mode():
        for i in range(len(df_use)):
            id_code = df_use.iloc[i]["id_code"]
            y = float(df_use.iloc[i]["diagnosis"])
            pth = os.path.join(train_img_dir, f"{id_code}.png")
            if not os.path.exists(pth):
                continue
            image = Image.open(pth).convert("RGB")
            image = image.resize((size, size), resample=Image.BILINEAR)
            x = norm(image).to(device)
            out = model(x.unsqueeze(0))
            out_flip = model(torch.flip(x.unsqueeze(0), dims=(3,)))
            pred = (out.item() + out_flip.item()) / 2.0
            if not np.isfinite(pred):
                continue
            xs.append(pred)
            ys.append(y)

    if len(xs) < 10:
        return 1.0, 0.0

    x = np.asarray(xs, dtype=np.float64)
    y = np.asarray(ys, dtype=np.float64)
    x_mean = x.mean()
    y_mean = y.mean()
    x_var = ((x - x_mean) ** 2).mean()
    if x_var <= 1e-12:
        return 1.0, y_mean - x_mean
    cov = ((x - x_mean) * (y - y_mean)).mean()
    a = cov / x_var
    b = y_mean - a * x_mean
    if not np.isfinite(a) or not np.isfinite(b):
        return 1.0, 0.0
    return float(a), float(b)


def _qwk_np(y_true, y_pred, n_classes=5):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    y_true = np.clip(y_true, 0, n_classes - 1)
    y_pred = np.clip(y_pred, 0, n_classes - 1)

    O = np.zeros((n_classes, n_classes), dtype=np.float64)
    for a, b in zip(y_true, y_pred):
        O[a, b] += 1.0

    act_hist = np.bincount(y_true, minlength=n_classes).astype(np.float64)
    pred_hist = np.bincount(y_pred, minlength=n_classes).astype(np.float64)
    E = np.outer(act_hist, pred_hist)
    if E.sum() > 0:
        E = E / E.sum() * O.sum()

    W = np.zeros((n_classes, n_classes), dtype=np.float64)
    denom = (n_classes - 1) ** 2
    for i in range(n_classes):
        for j in range(n_classes):
            W[i, j] = ((i - j) ** 2) / denom

    num = (W * O).sum()
    den = (W * E).sum()
    if den <= 0:
        return 0.0
    return 1.0 - num / den


def _apply_thresholds_continuous(preds_cont, thresholds):
    t0, t1, t2, t3 = thresholds
    preds_cont = np.asarray(preds_cont, dtype=np.float64)
    out = np.zeros_like(preds_cont, dtype=np.int64)
    out[preds_cont >= t0] = 1
    out[preds_cont >= t1] = 2
    out[preds_cont >= t2] = 3
    out[preds_cont >= t3] = 4
    return out


def _optimize_thresholds(
    model,
    train_df,
    train_img_dir,
    norm,
    size=256,
    device=device,
    max_images=384,
    a_cal=1.0,
    b_cal=0.0,
):
    default_thr = np.array([0.75, 1.5, 2.5, 3.5], dtype=np.float64)

    model.eval()
    xs, ys = [], []

    df_use = _stratified_sample_df(
        train_df, "diagnosis", n=min(max_images, len(train_df)), seed=2021
    )

    with torch.inference_mode():
        for i in range(len(df_use)):
            id_code = df_use.iloc[i]["id_code"]
            y = int(df_use.iloc[i]["diagnosis"])
            pth = os.path.join(train_img_dir, f"{id_code}.png")
            if not os.path.exists(pth):
                continue
            image = Image.open(pth).convert("RGB")
            image = image.resize((size, size), resample=Image.BILINEAR)
            x = norm(image).to(device)
            out = model(x.unsqueeze(0))
            out_flip = model(torch.flip(x.unsqueeze(0), dims=(3,)))
            pred = (out.item() + out_flip.item()) / 2.0
            if not np.isfinite(pred):
                continue

            pred = a_cal * float(pred) + b_cal
            if not np.isfinite(pred):
                continue

            xs.append(float(pred))
            ys.append(y)

    if len(xs) < 50:
        return default_thr.tolist()

    x = np.asarray(xs, dtype=np.float64)
    y = np.asarray(ys, dtype=np.int64)

    q_grid = np.linspace(0.02, 0.98, 49)
    cand_vals = np.unique(np.quantile(x, q_grid))
    cand_vals.sort()

    def qwk_for_thr(thr):
        yp = _apply_thresholds_continuous(x, thr)
        return _qwk_np(y, yp, n_classes=5)

    thr = default_thr.copy()
    for k in range(4):
        left = x[y <= k]
        right = x[y >= k + 1]
        if len(left) > 5 and len(right) > 5:
            thr[k] = 0.5 * (np.median(left) + np.median(right))
    thr.sort()

    best_thr = thr.copy()
    best_score = qwk_for_thr(best_thr)

    for _ in range(4):
        improved = False
        for k in range(4):
            lo = -np.inf if k == 0 else best_thr[k - 1] + 1e-6
            hi = np.inf if k == 3 else best_thr[k + 1] - 1e-6
            feasible = cand_vals[(cand_vals > lo) & (cand_vals < hi)]
            if feasible.size == 0:
                continue
            local_best_s = best_score
            local_best_v = best_thr[k]
            for v in feasible:
                cand = best_thr.copy()
                cand[k] = v
                s = qwk_for_thr(cand)
                if s > local_best_s + 1e-12:
                    local_best_s = s
                    local_best_v = v
            if local_best_s > best_score + 1e-12:
                best_score = local_best_s
                best_thr[k] = local_best_v
                improved = True
        if not improved:
            break

    best_thr = np.sort(best_thr)
    return best_thr.tolist()


def _fit_ridge_head_on_features(
    model,
    train_df,
    train_img_dir,
    norm,
    size=256,
    device=device,
    max_images=512,
    lam=10.0,
):
    """
    If no DR-specific checkpoint exists, the regressor head is random; fit it in closed-form on frozen features.
    Keeps architecture/inference semantics identical; avoids near-random outputs that hurt kappa.
    """
    model.eval()
    feats = []
    ys = []

    df_use = _stratified_sample_df(
        train_df, "diagnosis", n=min(max_images, len(train_df)), seed=2022
    )

    with torch.inference_mode():
        for i in range(len(df_use)):
            id_code = df_use.iloc[i]["id_code"]
            y = float(df_use.iloc[i]["diagnosis"])
            pth = os.path.join(train_img_dir, f"{id_code}.png")
            if not os.path.exists(pth):
                continue
            image = Image.open(pth).convert("RGB")
            image = image.resize((size, size), resample=Image.BILINEAR)
            x = norm(image).to(device)

            f = model.features(x.unsqueeze(0))
            f_flip = model.features(torch.flip(x.unsqueeze(0), dims=(3,)))
            f = 0.5 * (f + f_flip)

            f = F.relu(f, inplace=False)
            if hasattr(model, "avg_pool") and model.avg_pool is not None:
                f = model.avg_pool(f)
            else:
                f = F.adaptive_avg_pool2d(f, (1, 1))
            f = f.view(f.size(0), -1)

            fv = f.squeeze(0).detach().cpu().numpy().astype(np.float64)
            if not np.all(np.isfinite(fv)):
                continue
            feats.append(fv)
            ys.append(y)

    if len(feats) < 50:
        return False

    X = np.stack(feats, axis=0)  # (n, d)
    y = np.asarray(ys, dtype=np.float64)  # (n,)
    n_s, d = X.shape

    X1 = np.concatenate([X, np.ones((n_s, 1), dtype=np.float64)], axis=1)  # add bias
    I = np.eye(d + 1, dtype=np.float64)
    I[-1, -1] = 0.0  # do not penalize bias

    A = X1.T @ X1 + lam * I
    b = X1.T @ y
    try:
        w = np.linalg.solve(A, b)  # (d+1,)
    except np.linalg.LinAlgError:
        w = np.linalg.lstsq(A, b, rcond=None)[0]

    w_weight = w[:d].astype(np.float32)
    w_bias = float(w[d])

    with torch.no_grad():
        model.last_linear.weight.copy_(
            torch.from_numpy(w_weight).view(1, -1).to(model.last_linear.weight.device)
        )
        model.last_linear.bias.copy_(
            torch.tensor(
                [w_bias], dtype=torch.float32, device=model.last_linear.bias.device
            )
        )
    return True




In [7]:
sample_sub = pd.read_csv(
    "/kaggle/input/aptos2019-blindness-detection/sample_submission.csv"
)
train_df = pd.read_csv("/kaggle/input/aptos2019-blindness-detection/train.csv")

available_model_paths = [
    "../input/densenet121/model_densenet121_bs64_30.pth",
    "../input/seresnet50testpseudo/model10.pth",
    "../input/seresnet50-512/model30_512.pth",
]
available_model_paths = [p for p in available_model_paths if os.path.exists(p)]

if len(available_model_paths) == 0:
    discovered = _find_candidate_checkpoints()
    available_model_paths = discovered[:3]

final_predictions = None
chosen_thresholds = [0.75, 1.5, 2.5, 3.5]

TRAIN_IMAGE_PATH = "/kaggle/input/aptos2019-blindness-detection/train_images"

if len(available_model_paths) == 0:
    model = get_densenet121_gem(pretrain="imagenet")

    norm = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

    model.to(device)

    # Change (score-relevant, minimal): use more stratified images for ridge/affine/thresholds
    # to reduce calibration variance and better match QWK without altering model/inference semantics.
    _ = _fit_ridge_head_on_features(
        model=model,
        train_df=train_df,
        train_img_dir=TRAIN_IMAGE_PATH,
        norm=norm,
        size=256,
        device=device,
        max_images=1024,
        lam=10.0,
    )

    a_cal, b_cal = _fit_affine_calibration(
        model=model,
        train_df=train_df,
        train_img_dir=TRAIN_IMAGE_PATH,
        norm=norm,
        size=256,
        device=device,
        max_images=1536,
    )
    chosen_thresholds = _optimize_thresholds(
        model=model,
        train_df=train_df,
        train_img_dir=TRAIN_IMAGE_PATH,
        norm=norm,
        size=256,
        device=device,
        max_images=1536,
        a_cal=a_cal,
        b_cal=b_cal,
    )

    preds = make_predictions(model, test_images, norm, size=256, device=device)
    final_predictions = [(idc, a_cal * float(p) + b_cal) for idc, p in preds]
else:
    model_path = available_model_paths[0]
    if "densenet121" in model_path.lower():
        model = get_densenet121_gem(pretrain=False)
        norm = transforms.Compose([transforms.ToTensor()])
    else:
        model = get_se_resnet50_gem(pretrain=False)
        norm = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ]
        )

    model.to(device)

    ckpt = torch.load(model_path, map_location="cpu")
    if (
        isinstance(ckpt, dict)
        and "state_dict" in ckpt
        and isinstance(ckpt["state_dict"], dict)
    ):
        state_dict = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and all(isinstance(k, str) for k in ckpt.keys()):
        state_dict = ckpt
    else:
        state_dict = ckpt

    if isinstance(state_dict, dict) and any(
        k.startswith("module.") for k in state_dict.keys()
    ):
        state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict, strict=False)
    model.to(device)

    # Change (score-relevant, minimal): same rationale—more data stabilizes calibration/thresholds.
    a_cal, b_cal = _fit_affine_calibration(
        model=model,
        train_df=train_df,
        train_img_dir=TRAIN_IMAGE_PATH,
        norm=norm,
        size=256,
        device=device,
        max_images=1536,
    )

    chosen_thresholds = _optimize_thresholds(
        model=model,
        train_df=train_df,
        train_img_dir=TRAIN_IMAGE_PATH,
        norm=norm,
        size=256,
        device=device,
        max_images=2048,
        a_cal=a_cal,
        b_cal=b_cal,
    )

    preds = make_predictions(model, test_images, norm, size=256, device=device)
    final_predictions = [(idc, a_cal * float(p) + b_cal) for idc, p in preds]



Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 106MB/s]


In [8]:
if final_predictions is not None:
    submission = pd.DataFrame(final_predictions, columns=["id_code", "diagnosis"])

    submission = test_df[["id_code"]].merge(submission, on="id_code", how="left")
    submission["diagnosis"] = submission["diagnosis"].fillna(0.0)

    t0, t1, t2, t3 = chosen_thresholds

    submission.loc[submission.diagnosis < t0, "diagnosis"] = 0
    submission.loc[
        (t0 <= submission.diagnosis) & (submission.diagnosis < t1), "diagnosis"
    ] = 1
    submission.loc[
        (t1 <= submission.diagnosis) & (submission.diagnosis < t2), "diagnosis"
    ] = 2
    submission.loc[
        (t2 <= submission.diagnosis) & (submission.diagnosis < t3), "diagnosis"
    ] = 3
    submission.loc[t3 <= submission.diagnosis, "diagnosis"] = 4

    submission["diagnosis"] = submission["diagnosis"].astype(int)
    submission.to_csv("submission.csv", index=False)
else:
    submission = sample_sub.copy()
    submission["diagnosis"] = 0
    submission.to_csv("submission.csv", index=False)

submission.head()

,id_code,diagnosis
0,b460ca9fa26f,0
1,6cee2e148520,0
2,ca6842bfcbc9,1
3,6cbc3dad809c,2
4,a9bc2f892cb3,0
